- Codex : https://openai.com/ko-KR/codex/ 윈도우 설치 다운로드 실행
- 윈도우 검색창 >> Chat GPT
# [프롬프트 1]
    - 내 컴퓨터 바탕화면에 사과폴더 만들어줘
# [프롬프트 2]
    - 아래의 내용을 참조하여 GUI 기능을 파이썬 코드로 만들어줘
    - 입력
      - 사원번호, 이름
    - 버튼
      - 확인, 취소

In [6]:
import tkinter as tk
from tkinter import messagebox


class EmployeeInputApp(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("사원 정보 입력")
        self.resizable(False, False)
        self.configure(padx=24, pady=20)

        self.employee_number = tk.StringVar()
        self.employee_name = tk.StringVar()

        tk.Label(self, text="사원번호").grid(row=0, column=0, sticky="w", pady=(0, 8))
        number_entry = tk.Entry(self, textvariable=self.employee_number, width=28)
        number_entry.grid(row=0, column=1, padx=(16, 0), pady=(0, 8))

        tk.Label(self, text="이름").grid(row=1, column=0, sticky="w")
        tk.Entry(self, textvariable=self.employee_name, width=28).grid(
            row=1, column=1, padx=(16, 0)
        )

        button_frame = tk.Frame(self)
        button_frame.grid(row=2, column=0, columnspan=2, pady=(20, 0))
        tk.Button(button_frame, text="확인", width=12, command=self.confirm).grid(
            row=0, column=0, padx=(0, 8)
        )
        tk.Button(button_frame, text="취소", width=12, command=self.cancel).grid(
            row=0, column=1
        )
        number_entry.focus()

    def confirm(self):
        number = self.employee_number.get().strip()
        name = self.employee_name.get().strip()

        if not number or not name:
            messagebox.showwarning("입력 확인", "사원번호와 이름을 모두 입력하세요.")
            return

        messagebox.showinfo(
            "확인", f"입력한 사원 정보입니다.\n\n사원번호: {number}\n이름: {name}"
        )

    def cancel(self):
        self.employee_number.set("")
        self.employee_name.set("")


if __name__ == "__main__":
    EmployeeInputApp().mainloop()

# [프롬프트 3] 
1. 내가 첨부한 05_Agent.ipynb를 파악해줘
2. 위의 코드를 기반으로 아래의 기능을 추가하여 발전한 파이썬 코드를 제작해줘
    - 2-1) GUI 기능 추가(수신이메일/앱비밀번호 사용자 세팅 & 체크박스를 통해서 저장할 수 있게)
    - 2-2) D:\Edu\Machine Learning\workspace\20.업무자동화\codex결과.ipynb 파일 작성 후 저장해줘

In [7]:
# 필요한 패키지 설치 (처음 한 번만 실행)
# %pip install selenium pandas openpyxl

import json
import re
import smtplib
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from email.message import EmailMessage
from pathlib import Path
import tkinter as tk
from tkinter import messagebox, scrolledtext, ttk

import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

BASE_DIR = Path.cwd()
CONFIG_PATH = BASE_DIR / "agent_email_settings.json"
DEFAULT_KEYWORDS = ["에어컨", "선풍기", "제습기", "냉풍기", "서큘레이터"]


def naver_cafe_search(keyword, wait_seconds=10):
    """키워드 하나의 네이버 검색 결과를 수집합니다."""
    driver = webdriver.Chrome()
    result = []

    try:
        driver.get(
            "https://search.naver.com/search.naver?where=view&query=" + keyword
        )

        WebDriverWait(driver, wait_seconds).until(
            EC.presence_of_all_elements_located(
                (By.CSS_SELECTOR, "a.title_link")
            )
        )

        for element in driver.find_elements(By.CSS_SELECTOR, "a.title_link"):
            title = element.text.strip()
            url = element.get_attribute("href")

            if title and url:
                result.append({
                    "키워드": keyword,
                    "게시글제목": title,
                    "URL": url
                })

    except Exception as error:
        print(f"[{keyword}] 검색 오류: {error}")

    finally:
        driver.quit()

    return result


def search_all(keywords, progress):
    """여러 키워드를 병렬로 검색합니다."""
    keywords = [item.strip() for item in keywords if item.strip()]

    if not keywords:
        raise ValueError("검색 키워드를 하나 이상 입력하세요.")

    rows = []

    with ThreadPoolExecutor(max_workers=min(3, len(keywords))) as executor:
        futures = {
            executor.submit(naver_cafe_search, keyword): keyword
            for keyword in keywords
        }

        for future in as_completed(futures):
            keyword = futures[future]
            rows.extend(future.result())
            progress(f"{keyword} 검색 완료")

    return rows


def save_excel(rows):
    """검색 결과를 Excel 파일로 저장합니다."""
    output_dir = BASE_DIR / "결과" / "웹"
    output_dir.mkdir(parents=True, exist_ok=True)

    file_path = output_dir / (
        f"naver_cafe_{datetime.now():%Y%m%d_%H%M%S}.xlsx"
    )

    df = pd.DataFrame(
        rows,
        columns=["키워드", "게시글제목", "URL"]
    )
    df.to_excel(file_path, index=False)

    return file_path


def send_result_email(sender, recipient, app_password, attachment):
    """Gmail 앱 비밀번호로 검색 결과 Excel을 전송합니다."""
    if not all([sender, recipient, app_password]):
        raise ValueError(
            "발신 Gmail, 수신 이메일, 앱 비밀번호를 모두 입력하세요."
        )

    if not re.fullmatch(r"[^@\s]+@[^@\s]+\.[^@\s]+", recipient):
        raise ValueError("수신 이메일 형식을 확인하세요.")

    message = EmailMessage()
    message["Subject"] = "네이버 카페 검색 결과"
    message["From"] = sender
    message["To"] = recipient
    message.set_content("네이버 카페 검색 결과 Excel 파일을 첨부합니다.")

    with open(attachment, "rb") as file:
        message.add_attachment(
            file.read(),
            maintype="application",
            subtype="vnd.openxmlformats-officedocument.spreadsheetml.sheet",
            filename=attachment.name
        )

    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as smtp:
        smtp.login(sender, app_password.replace(" ", ""))
        smtp.send_message(message)


class AgentApp:
    def __init__(self, root):
        self.root = root
        root.title("네이버 카페 검색 Agent")
        root.geometry("650x560")

        try:
            saved = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
        except (FileNotFoundError, json.JSONDecodeError):
            saved = {}

        self.sender = tk.StringVar(
            value=saved.get("sender_email", "")
        )
        self.recipient = tk.StringVar(
            value=saved.get("recipient_email", "")
        )
        self.password = tk.StringVar(
            value=saved.get("app_password", "")
        )
        self.save = tk.BooleanVar(
            value=saved.get("save_settings", False)
        )

        frame = ttk.Frame(root, padding=16)
        frame.pack(fill="both", expand=True)
        frame.columnconfigure(1, weight=1)

        for row, (label, variable, mask) in enumerate([
            ("발신 Gmail", self.sender, None),
            ("수신 이메일", self.recipient, None),
            ("Gmail 앱 비밀번호", self.password, "*"),
        ]):
            ttk.Label(frame, text=label).grid(
                row=row, column=0, sticky="w",
                padx=(0, 10), pady=5
            )
            ttk.Entry(
                frame,
                textvariable=variable,
                show=mask
            ).grid(
                row=row, column=1, sticky="ew", pady=5
            )

        ttk.Checkbutton(
            frame,
            text="이메일 설정 저장",
            variable=self.save
        ).grid(row=3, column=1, sticky="w")

        ttk.Label(
            frame,
            text="검색 키워드 (쉼표로 구분)"
        ).grid(
            row=4, column=0, sticky="nw",
            padx=(0, 10), pady=8
        )

        self.keywords = tk.Text(frame, height=4)
        self.keywords.grid(row=4, column=1, sticky="ew", pady=8)
        self.keywords.insert("1.0", ", ".join(DEFAULT_KEYWORDS))

        self.run_button = ttk.Button(
            frame,
            text="검색 후 Excel 저장 및 이메일 전송",
            command=self.start
        )
        self.run_button.grid(row=5, column=1, sticky="ew", pady=8)

        self.log = scrolledtext.ScrolledText(
            frame,
            height=14,
            state="disabled"
        )
        self.log.grid(
            row=6,
            column=0,
            columnspan=2,
            sticky="nsew"
        )
        frame.rowconfigure(6, weight=1)

    def write_log(self, text):
        self.root.after(0, lambda: self._write_log(text))

    def _write_log(self, text):
        self.log.configure(state="normal")
        self.log.insert("end", text + "\n")
        self.log.see("end")
        self.log.configure(state="disabled")

    def save_settings(self):
        if self.save.get():
            CONFIG_PATH.write_text(
                json.dumps({
                    "sender_email": self.sender.get().strip(),
                    "recipient_email": self.recipient.get().strip(),
                    "app_password": self.password.get(),
                    "save_settings": True
                }, ensure_ascii=False, indent=2),
                encoding="utf-8"
            )

        elif CONFIG_PATH.exists():
            CONFIG_PATH.unlink()

    def start(self):
        self.save_settings()
        self.run_button.configure(state="disabled")

        threading.Thread(
            target=self.run_agent,
            daemon=True
        ).start()

    def run_agent(self):
        try:
            self.write_log("검색을 시작합니다...")

            rows = search_all(
                self.keywords.get("1.0", "end").split(","),
                self.write_log
            )

            excel_path = save_excel(rows)
            self.write_log(f"Excel 저장 완료: {excel_path}")

            send_result_email(
                self.sender.get().strip(),
                self.recipient.get().strip(),
                self.password.get(),
                excel_path
            )

            self.write_log("이메일 전송 완료")

            self.root.after(
                0,
                lambda: messagebox.showinfo(
                    "완료",
                    f"처리가 완료되었습니다.\n{excel_path}"
                )
            )

        except Exception as error:
            self.write_log(f"오류: {error}")

            self.root.after(
                0,
                lambda: messagebox.showerror(
                    "오류",
                    str(error)
                )
            )

        finally:
            self.root.after(
                0,
                lambda: self.run_button.configure(state="normal")
            )


root = tk.Tk()
AgentApp(root)
root.mainloop()

[선풍기] 검색 오류: Message: 

[제습기] 검색 오류: Message: 

[에어컨] 검색 오류: Message: 

[냉풍기] 검색 오류: Message: 

[서큘레이터] 검색 오류: Message: 



In [ ]:
# End of -------------------------------------------------